Loading and merging the dataset

In [102]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
file_path = r"E:\R_Thesis\SAS_2024F.xlsx"

coordinates = pd.read_excel(file_path, sheet_name="Coordinates")
season_a = pd.read_excel(file_path, sheet_name="SeasonA")
season_b = pd.read_excel(file_path, sheet_name="SeasonB")
season_c = pd.read_excel(file_path, sheet_name="SeasonC")

print(season_a.shape, season_b.shape, season_c.shape)

(39330, 60) (35584, 67) (1949, 61)


In [90]:
season_all = pd.concat([season_a, season_b, season_c], ignore_index=True)
print("Combined data:", season_all.shape)

Combined data: (76863, 81)


In [91]:
data = season_all.merge(coordinates, on="District_Cl", how="left")

print("Merged data:", data.shape)
print("Missing latitude:", data["latitude"].isna().sum())

Merged data: (76863, 84)
Missing latitude: 0


Exploration of the dataset

In [94]:
farmer_crop = pd.crosstab(
    data["1.7 Farmer/LSF type"],
    data["Q.2.4 Crop name.1"]
)
print(farmer_crop)

Q.2.4 Crop name.1                                   Banana for beer  \
1.7 Farmer/LSF type                                                   
Large scale  farmer as Coperative/Company/Assoc...                5   
Large scale farmer as individual                                 42   
Small Scale farmer as Coperative/Company/Associ...                6   
Small scale farmer as individual                               6265   

Q.2.4 Crop name.1                                   Bush bean  Cassava  \
1.7 Farmer/LSF type                                                      
Large scale  farmer as Coperative/Company/Assoc...        100        6   
Large scale farmer as individual                          297       36   
Small Scale farmer as Coperative/Company/Associ...         21        8   
Small scale farmer as individual                        10528     7225   

Q.2.4 Crop name.1                                   Climbing bean  \
1.7 Farmer/LSF type                                        

In [95]:
farmer_counts = data["1.7 Farmer/LSF type"].value_counts()
farmer_pct = data["1.7 Farmer/LSF type"].value_counts(normalize=True) * 100

farmer_summary = pd.DataFrame({
    "Count": farmer_counts,
    "Percentage (%)": farmer_pct.round(2)
})

print(farmer_summary)

                                                    Count  Percentage (%)
1.7 Farmer/LSF type                                                      
Small scale farmer as individual                    73684           95.86
Large scale farmer as individual                     1635            2.13
Large scale  farmer as Coperative/Company/Assoc...   1349            1.76
Small Scale farmer as Coperative/Company/Associ...    195            0.25


In [96]:
crop_counts = data["Q.2.4 Crop name.1"].value_counts()
crop_pct = data["Q.2.4 Crop name.1"].value_counts(normalize=True) * 100

crop_summary = pd.DataFrame({
    "Count": crop_counts,
    "Percentage (%)": crop_pct.round(2)
})

print(crop_summary)

# Top 3 crops
print("Top 3 crops:")
print(crop_summary.head(3))

                   Count  Percentage (%)
Q.2.4 Crop name.1                       
Maize              11464           14.91
Bush bean          10946           14.24
Cassava             7275            9.46
Climbing bean       6371            8.29
Sweet potato        6368            8.28
Banana for beer     6318            8.22
Cooking banana      5622            7.31
Dessert banana      4994            6.50
Sorghum             3021            3.93
Irish potato        2974            3.87
Soybean             2413            3.14
Vegetables          1906            2.48
Taro & Yams         1885            2.45
Other crops         1265            1.65
Groundnut           1165            1.52
Pea                  885            1.15
Fodder crops         570            0.74
Fruits               465            0.60
Paddy rice           420            0.55
Other cereals        342            0.44
Wheat                194            0.25
Top 3 crops:
                   Count  Percentage (%)
Q.2

In [103]:
vegetables = [
    "Tomato", "Cabbage", "Onion", "Carrot",
    "Green pepper", "Eggplant", "Spinach"
]

In [104]:
veg = data[data["Q.2.4 Crop name.1"] == "Vegetables"]

print("Vegetables observations:", len(veg))
print("\nVegetable share (%):")
print(len(veg) / len(data) * 100)

Vegetables observations: 1906

Vegetable share (%):
2.4797366743426617


In [105]:
top5 = data["Q.2.4 Crop name.1"].value_counts().head(5).index.tolist()
key_crops = top5 + ["Vegetables", "Fruits"]
print(top5)

['Maize', 'Bush bean', 'Cassava', 'Climbing bean', 'Sweet potato']


In [106]:
key_crops = top5 + ["Vegetables", "Fruits"]

In [107]:
print(key_crops )

['Maize', 'Bush bean', 'Cassava', 'Climbing bean', 'Sweet potato', 'Vegetables', 'Fruits']


Storage  and market adoption

In [115]:
storage_col = "Q.2.37 What is the storage facility used during this agricultural season?"
market_col = "Q.2.27 On which market this crop was sold?"

In [116]:
data["storage_adoption"] = data[storage_col].notna().astype(int)

In [117]:
summary = data.groupby("Q.2.4 Crop name.1").agg(
    storage_adoption=("storage_adoption", "mean"),
    market_users=(market_col, lambda x: x.notna().mean())
) * 100

summary = summary.sort_values("storage_adoption", ascending=False)

print(summary.round(2))

                   storage_adoption  market_users
Q.2.4 Crop name.1                                
Bush bean                      8.36         35.23
Maize                          7.17         42.46
Climbing bean                  6.78         30.25
Wheat                          5.15         54.12
Soybean                        4.43         31.45
Other cereals                  4.39         31.58
Sorghum                        4.27         63.75
Groundnut                      3.78         42.83
Fodder crops                   1.05          3.51
Pea                            1.02         37.29
Cassava                        0.74         22.97
Irish potato                   0.71         55.51
Sweet potato                   0.28         40.56
Vegetables                     0.10         80.48
Cooking banana                 0.07         37.60
Dessert banana                 0.02         70.89
Taro & Yams                    0.00         23.87
Other crops                    0.00         78.81


Variable construction

In [134]:
# Outcome variable "Binary indicator of PHL occurrence"
data["PHL_binary"] = (data["total_loss"] > 0).astype(int)

# Continuous PHL rate
data["PHL_rate"] = data["total_loss"] / data["gross_production"]

# Clean extreme and missing values
data["PHL_rate"] = (
    data["PHL_rate"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
    .clip(0, 1)
)

In [133]:
##Treatment variables
data["Storage_access"] = data[
    "Q.2.37 What is the storage facility used during this agricultural season?"
].apply(lambda x: 0 if str(x).lower() in ["none", "no", "nan"] else 1)

In [124]:
formal_markets = ["Market", "Cooperative", "Company", "Association"]

data["Market_access"] = data[
    "Q.2.27 On which market this crop was sold?"
].apply(lambda x: 1 if str(x) in formal_markets else 0)

In [135]:
## Control variables
X_core = data[[
    "1.8 Gender",
    "1.9 Age",
    "2.2 Plot area(sqm)",
    "District_Cl"
]]

In [126]:
X_practices = data[[
    "4.15 Has this plot been irrigated during this agricultural season?",
    "3.3 Have you used organic fertilizer in this plot during this season?",
    "3.9 Have you used inorganic fertilizer in this plot during this season?",
    "3.18 Did you use pesticide/Fungicide in any of your plots during this season?"
]]

In [127]:
X_farm = data[[
    "4.1 What is the degree of erosion on this plot?",
    "4.6 Is this plot located in land consolidated site in this season?"
]]

In [128]:
X = pd.concat([X_core, X_practices, X_farm], axis=1)

In [131]:
## Variables"Outcome, Treatment, and Controls"
Y = data["PHL_binary"]        # Outcome
D = data["Storage_access"]    # Treatment 1
W = data["Market_access"]     # Treatment 2 / control factor

In [132]:
##Final dataset.shape
ml_data = pd.concat([Y, D, W, X], axis=1)

print("Before drop NA:", ml_data.shape)

ml_data = ml_data.dropna()

print("After drop NA:", ml_data.shape)

Before drop NA: (76863, 13)
After drop NA: (2811, 13)


In [136]:
ml_data = pd.concat([Y, D, W, X], axis=1)

In [137]:
ml_data = ml_data.dropna()

In [138]:
ml_data = pd.concat([Y, D, W, X], axis=1)

print("Dataset ready for analysis:", ml_data.shape)

Dataset ready for analysis: (76863, 13)


Data cleaning

In [153]:
veg_data = data[data["Q.2.4 Crop name.1"] == "Vegetables"].copy()

In [154]:
## Missing numbers cleaning
veg_data.isna().sum()

IDQUEST                      0
1.1 Province                 0
1.2 District name & code     0
District_Cl                  0
1.3 Stratum                  0
                            ..
gross_production            28
PHL_binary                   0
PHL_rate                     0
Market_access                0
Storage_access               0
Length: 91, dtype: int64

In [155]:
data["Q.2.4 Crop name"]

0        Napia grass for fodder
1              Maize for fodder
2                        Mucuna
3            Soybean for fodder
4            Soybean for fodder
                  ...          
76858                  Eggplant
76859                  Eggplant
76860                  Amaranth
76861                  Eggplant
76862              Sweet potato
Name: Q.2.4 Crop name, Length: 76863, dtype: object

In [170]:
veg_data = data[
    data["Q.2.4 Crop name.1"]
    .str.strip()
    .str.lower() == "vegetables"
].copy()

print("Vegetable data shape:", veg_data.shape)

Vegetable data shape: (1906, 91)


In [171]:
missing = veg_data.isna().sum().sort_values(ascending=False)
print(missing.head(15))

Unnamed: 57                                                                         1906
Unnamed: 54                                                                         1906
Unnamed: 56                                                                         1906
Unnamed: 58                                                                         1906
Unnamed: 55                                                                         1906
Q.2.37 What is the storage facility used during this agricultural season?           1904
Q.2.39 Qty of this crop what is the quantity that has been lost?.1                  1902
4.9 Did you use any mechanical equipment for agriculture activities on this plot    1888
s2q24_c                                                                             1766
s2q24_b                                                                             1414
Plot_weight                                                                         1338
4.2 Is there any anti

In [172]:
veg_data = veg_data.dropna(subset=[
    "total_loss",
    "gross_production"
])

In [173]:
storage_col = "Q.2.37 What is the storage facility used during this agricultural season?"

veg_data["Storage_access"] = veg_data[storage_col].notna().astype(int)

In [174]:
market_col = "Q.2.27 On which market this crop was sold?"

formal_markets = ["Market", "Cooperative", "Company", "Association"]

veg_data["Market_access"] = veg_data[market_col].apply(
    lambda x: 1 if str(x) in formal_markets else 0
)

In [175]:
veg_data["PHL_binary"] = (veg_data["total_loss"] > 0).astype(int)

veg_data["PHL_rate"] = veg_data["total_loss"] / veg_data["gross_production"]

veg_data["PHL_rate"] = (
    veg_data["PHL_rate"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
    .clip(0, 1)
)

In [176]:
##Duplicates cleaning
veg_data = veg_data.drop_duplicates()

In [177]:
## Outlier cleaning
veg_data = veg_data[
    veg_data["PHL_rate"] <= veg_data["PHL_rate"].quantile(0.99)
]

In [178]:
print("Final vegetable dataset:", veg_data.shape)

veg_data[[
    "PHL_rate",
    "Storage_access",
    "Market_access"
]].describe()

Final vegetable dataset: (1859, 91)


,PHL_rate,Storage_access,Market_access
count,1859.000000,1859.000000,1859.000000
mean,0.008732,0.001076,0.516407
std,0.033133,0.032791,0.499865
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,1.000000
75%,0.000000,0.000000,1.000000
max,0.320000,1.000000,1.000000


Standardization

In [179]:
Y = veg_data["PHL_binary"]
D = veg_data["Storage_access"]
W = veg_data["Market_access"]

X = veg_data[[
    "1.9 Age",
    "2.2 Plot area(sqm)"
]]

In [180]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [182]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Step 1: Impute missing values
imputer = SimpleImputer(strategy="mean")
X_imputed = imputer.fit_transform(X)

# Step 2: Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

In [184]:
##Laso variable selection
from sklearn.linear_model import LassoCV

lasso = LassoCV(cv=5)
lasso.fit(X_scaled, Y)

selected = X.columns[lasso.coef_ != 0]
print("Selected variables:", selected)

Selected variables: Index(['1.9 Age', '2.2 Plot area(sqm)'], dtype='object')


In [186]:
X_selected = veg_data[['1.9 Age', '2.2 Plot area(sqm)']]

In [207]:
X_final = veg_data[[
    "1.9 Age",
    "2.2 Plot area(sqm)",
    "4.15 Has this plot been irrigated during this agricultural season?",
    "3.3 Have you used organic fertilizer in this plot during this season?",
    "3.9 Have you used inorganic fertilizer in this plot during this season?"
]]

In [231]:
X = X_final
Y = veg_data["PHL_binary"]

In [232]:
from sklearn.model_selection import train_test_split

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

In [233]:
from sklearn.preprocessing import StandardScaler

# Convert Yes/No before scaling (if not already done)
for col in X_train_raw.columns:
    if X_train_raw[col].dtype == "object":
        X_train_raw[col] = X_train_raw[col].map({
            "Yes": 1, "No": 0,
            "Male": 1, "Female": 0
        })
        X_test_raw[col] = X_test_raw[col].map({
            "Yes": 1, "No": 0,
            "Male": 1, "Female": 0
        })

In [234]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")

X_train = imputer.fit_transform(X_train_raw)
X_test = imputer.transform(X_test_raw)

In [235]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [236]:
logit.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [237]:
logit_pred = logit.predict(X_test)

evaluate_model("Logit", y_test, logit_pred)


Logit Accuracy: 0.8763440860215054
[[326   0]
 [ 46   0]]
              precision    recall  f1-score   support

           0       0.88      1.00      0.93       326
           1       0.00      0.00      0.00        46

    accuracy                           0.88       372
   macro avg       0.44      0.50      0.47       372
weighted avg       0.77      0.88      0.82       372



In [251]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import numpy as np

def evaluate_model(name, model, X_test, y_test, threshold=0.5):
    
    probs = model.predict_proba(X_test)[:, 1]
    preds = (probs >= threshold).astype(int)

    acc = accuracy_score(y_test, preds)
    cm = confusion_matrix(y_test, preds)

    print("\n======================")
    print(name)
    print("Accuracy:", acc)
    print(cm)
    print(classification_report(y_test, preds))

In [252]:
thresholds = [0.3, 0.4, 0.5, 0.6]

for t in thresholds:
    print("\nTHRESHOLD =", t)
    evaluate_model("Logit", logit, X_test, y_test, threshold=t)


THRESHOLD = 0.3

Logit
Accuracy: 0.9280556820399402
[[14250    19]
 [ 1087    17]]
              precision    recall  f1-score   support

           0       0.93      1.00      0.96     14269
           1       0.47      0.02      0.03      1104

    accuracy                           0.93     15373
   macro avg       0.70      0.51      0.50     15373
weighted avg       0.90      0.93      0.90     15373


THRESHOLD = 0.4

Logit
Accuracy: 0.9285110258244975
[[14267     2]
 [ 1097     7]]
              precision    recall  f1-score   support

           0       0.93      1.00      0.96     14269
           1       0.78      0.01      0.01      1104

    accuracy                           0.93     15373
   macro avg       0.85      0.50      0.49     15373
weighted avg       0.92      0.93      0.89     15373


THRESHOLD = 0.5

Logit
Accuracy: 0.9285760749365771
[[14268     1]
 [ 1097     7]]
              precision    recall  f1-score   support

           0       0.93      1.00      

In [254]:
rf_probs = rf.predict_proba(X_test)[:, 1]

for t in [0.3, 0.4, 0.5,0.7]:
    preds = (rf_probs >= t).astype(int)

    print("\nRF THRESHOLD:", t)
    print("Accuracy:", accuracy_score(y_test, preds))
    print(confusion_matrix(y_test, preds))


RF THRESHOLD: 0.3
Accuracy: 0.1717296558901971
[[ 1601 12668]
 [   65  1039]]

RF THRESHOLD: 0.4
Accuracy: 0.3787809796396279
[[4952 9317]
 [ 233  871]]

RF THRESHOLD: 0.5
Accuracy: 0.7587978924087686
[[11175  3094]
 [  614   490]]

RF THRESHOLD: 0.7
Accuracy: 0.9244779808755611
[[14104   165]
 [  996   108]]
